# Try EazyDataFix

This notebook installs the published package, profiles a small messy sales file from the repository, previews a v2 recipe, applies it, saves the recipe, and replays that recipe on a second file.

`preview` does not change the source file. The recipe declares types. It does not train a model or build a Power BI report.

The sample uses Indian currency and day-first dates. An internal quality check may warn about that format. The cell that downloads the file hides the repeated warning. Conversion still uses `date_format="%d/%m/%Y"`.

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "eazydatafix"])
print("Installed eazydatafix")


Installed eazydatafix


In [2]:
import urllib.request
import warnings
from pathlib import Path

import eazydatafix as edf

warnings.filterwarnings(
    "ignore",
    message=r"Parsing dates in %d/%m/%Y format when dayfirst=False.*",
    category=UserWarning,
)

url = "https://raw.githubusercontent.com/suneelprojects/eazydatafix/main/examples/v2_sales.csv"
sales_path = Path("v2_sales.csv")
sales_path.write_bytes(urllib.request.urlopen(url, timeout=30).read())
print(sales_path.read_text(encoding="utf-8"))

Customer ID,Amount,Order Date,City
001,"₹1,250.50",01/02/2025,HYD
002,₹800,02/02/2025,BLR
003,"₹2,300",03/02/2025,Hyderabad



In [3]:
profile = edf.profile(sales_path)
print(f"{profile.file_name}: {profile.rows} rows, {profile.columns} columns")
for name, data_type in zip(profile.column_names, profile.data_types):
    label = getattr(data_type, "value", data_type)
    print(f"- {name}: {label}")

v2_sales.csv: 3 rows, 4 columns
- Customer ID: Integer
- Amount: Text
- Order Date: Text
- City: Text


In [4]:
recipe = edf.PipelineRecipe(
    name="monthly_sales",
    columns={
        "customer_id": edf.ColumnRule(dtype="string"),
        "amount": edf.ColumnRule(dtype="number", currency_symbols=("₹",)),
        "order_date": edf.ColumnRule(dtype="datetime", date_format="%d/%m/%Y"),
        "city": edf.ColumnRule(
            dtype="string",
            category_mapping={"HYD": "Hyderabad", "BLR": "Bengaluru"},
        ),
    },
)
pipeline = edf.Pipeline(recipe)
plan = pipeline.preview(sales_path)
print("Preview ready:", plan.result.is_ready)
print(plan.result.dataset.to_string(index=False))

Preview ready: True
customer_id  amount order_date      city
        001  1250.5 2025-02-01 Hyderabad
        002   800.0 2025-02-02 Bengaluru
        003  2300.0 2025-02-03 Hyderabad


In [5]:
result = pipeline.apply(sales_path, plan)
print("Applied ready:", result.is_ready)
print("Recorded transformations:", len(result.changes))
print(result.dataset.dtypes.astype(str).to_string())

Applied ready: True
Recorded transformations: 6
customer_id            string
amount                Float64
order_date     datetime64[us]
city                   string


In [6]:
recipe_path = Path("monthly_sales.json")
recipe.save(recipe_path)
second_path = Path("v2_sales_replay.csv")
second_path.write_text(
    "Customer ID,Amount,Order Date,City\n"
    '004,"₹500.00",04/02/2025,HYD\n'
    "005,₹75,05/02/2025,BLR\n",
    encoding="utf-8",
)
replayed = edf.Pipeline(edf.PipelineRecipe.load(recipe_path)).run(second_path)
print("Replayed ready:", replayed.is_ready)
print(replayed.dataset.to_string(index=False))

Replayed ready: True
customer_id  amount order_date      city
        004   500.0 2025-02-04 Hyderabad
        005    75.0 2025-02-05 Bengaluru
